# 05 — Test du modèle (pipeline refactorisé)

Ce notebook sert de **smoke test** pour vérifier rapidement que le pipeline fonctionne avec la nouvelle structure :

- chargement du dataset
- transforms
- DataLoader
- construction du modèle
- forward pass
- validation rapide
- chargement d'un checkpoint si disponible

Il est compatible avec la structure :

```text
data/raw/brain_mri/
src/brain_tumor_mri/
artifacts/checkpoints/
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from brain_tumor_mri.data.dataset import BrainMRIDataset
from brain_tumor_mri.data.transforms import get_eval_transforms
from brain_tumor_mri.data.loaders import make_loaders
from brain_tumor_mri.models.builder import build_model
from brain_tumor_mri.training.engine import validate_one_epoch, predict_probabilities
from brain_tumor_mri.evaluation.metrics import compute_metrics, compute_auprc
from brain_tumor_mri.utils import get_device, load_checkpoint

print("Torch version:", torch.__version__)

## 1. Configuration

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

MODEL_NAME = "resnet18"
NUM_CLASSES = 2
PRETRAINED = False  # False ici pour tester rapidement le pipeline

DATA_DIR = Path("data/raw/brain_mri/Testing")
CHECKPOINT_PATH = Path("artifacts/checkpoints/best_resnet18_train_testing_split.pt")

## 2. Device

In [ ]:
device = get_device()
print("Device:", device)

## 3. Dataset et DataLoader

In [ ]:
eval_tfms = get_eval_transforms(img_size=IMG_SIZE)

dataset = BrainMRIDataset(DATA_DIR, transform=eval_tfms)

print("Dataset size:", len(dataset))
print("First 5 labels:", dataset.get_labels()[:5])

In [ ]:
loader, _ = make_loaders(
    dataset,
    dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    weighted=False,
    pin_memory=False,
)

print("Number of batches:", len(loader))

## 4. Visualisation rapide

In [ ]:
raw_dataset = BrainMRIDataset(DATA_DIR, transform=None)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for i, ax in enumerate(axes):
    img, label = raw_dataset[i]
    ax.imshow(img)
    ax.set_title(f"Label: {label}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Construction du modèle

In [ ]:
model = build_model(
    model_name=MODEL_NAME,
    pretrained=PRETRAINED,
    num_classes=NUM_CLASSES,
    img_size=IMG_SIZE,
)
model.to(device)

print(model.__class__.__name__)

## 6. Forward pass sur un batch

In [ ]:
images, labels = next(iter(loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Input batch shape :", images.shape)
print("Labels shape      :", labels.shape)
print("Output shape      :", outputs.shape)

## 7. Validation rapide avec modèle initial

In [ ]:
criterion = nn.CrossEntropyLoss()

stats = validate_one_epoch(
    model=model,
    loader=loader,
    criterion=criterion,
    device=device,
)

print("Loss:", round(stats["loss"], 4))
print("Accuracy:", round(stats["acc"], 4))

## 8. Chargement du checkpoint (si disponible)

Cette cellule recharge le meilleur modèle entraîné si le checkpoint existe.


In [ ]:
if CHECKPOINT_PATH.exists():
    model = build_model(
        model_name=MODEL_NAME,
        pretrained=False,
        num_classes=NUM_CLASSES,
        img_size=IMG_SIZE,
    )
    model = load_checkpoint(model, CHECKPOINT_PATH, map_location=device)
    model.to(device)
    print(f"Checkpoint chargé depuis : {CHECKPOINT_PATH}")
else:
    print(f"Aucun checkpoint trouvé : {CHECKPOINT_PATH}")

## 9. Évaluation sur le checkpoint

In [ ]:
criterion = nn.CrossEntropyLoss()

stats = validate_one_epoch(
    model=model,
    loader=loader,
    criterion=criterion,
    device=device,
)

metrics = compute_metrics(stats["targets"], stats["preds"])
auprc = compute_auprc(stats["targets"], stats["probs"])

print(f"Loss      : {stats['loss']:.4f}")
print(f"Accuracy  : {metrics['accuracy']:.4f}")
print(f"Precision : {metrics['precision']:.4f}")
print(f"Recall    : {metrics['recall']:.4f}")
print(f"F1-score  : {metrics['f1']:.4f}")
print(f"PRAUC     : {auprc:.4f}")

## 10. Classification report et matrice de confusion

In [ ]:
print(metrics["classification_report"])
print(metrics["confusion_matrix"])

## 11. Probabilités prédites

Cette cellule est utile pour préparer ensuite :
- threshold tuning
- TTA
- analyse d'erreurs


In [ ]:
y_true, probs = predict_probabilities(model, loader, device)

print("Nombre de probabilités :", len(probs))
print("Premières probabilités :", probs[:10])

## 12. Histogramme des probabilités

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(probs, bins=20)
plt.title("Distribution des probabilités prédites")
plt.xlabel("P(class=1)")
plt.ylabel("Count")
plt.show()